# Midnight Expressway — Build APK (Fullscreen)
Chạy lần lượt các cell bên dưới để tạo icon cho game, clone dự án từ GitHub, cấu hình toàn màn hình (ẩn thanh trạng thái & thanh điều hướng), build `app-debug.apk` và tải về máy.

In [ ]:
import os, shutil, subprocess
from PIL import Image, ImageDraw

W, H = 1024, 1024
img = Image.new("RGBA", (W, H), (9, 11, 26, 255))
draw = ImageDraw.Draw(img)

# 1. Bầu trời đêm chuyển sắc
for y in range(H):
    t = y / H
    if t < 0.56:
        k = t / 0.56
        r = int(6 + 32 * k)
        g = int(8 + 10 * k)
        b = int(22 + 52 * k)
    else:
        r, g, b = 10, 12, 28
    draw.line([(0, y), (W, y)], fill=(r, g, b, 255))

# 2. Trăng tròn / mặt trời neon ở đường chân trời
cx, cy, rad = W // 2, 430, 185
for i in range(60, 0, -4):
    alpha = int(10 * (1 - i / 60))
    draw.ellipse([cx - rad - i, cy - rad - i, cx + rad + i, cy + rad + i], fill=(255, 60, 140, alpha))
for y in range(cy - rad, cy + rad):
    dy = y - cy
    dx = int((rad * rad - dy * dy) ** 0.5)
    k = (y - (cy - rad)) / (2 * rad)
    r = 255
    g = int(215 - 140 * k)
    b = int(105 + 45 * k)
    if y > cy + 25 and (y // 18) % 2 == 0:
        continue
    draw.line([(cx - dx, y), (cx + dx, y)], fill=(r, g, b, 255))

# 3. Đường cao tốc phối cảnh
hy = 560
draw.polygon([(cx - 46, hy), (cx + 46, hy), (W - 60, H), (60, H)], fill=(18, 20, 38, 255))
draw.line([(cx - 46, hy), (60, H)], fill=(0, 245, 212, 255), width=14)
draw.line([(cx + 46, hy), (W - 60, H)], fill=(255, 42, 133, 255), width=14)

# Vạch kẻ đường giữa
for i in range(6):
    y1 = hy + 20 + i * 75
    y2 = y1 + 38
    w = 4 + i * 2
    if y2 < H:
        draw.polygon([(cx - w, y1), (cx + w, y1), (cx + w + 2, y2), (cx - w - 2, y2)], fill=(255, 225, 110, 255))

# 4. Bóng xe thể thao & đèn hậu neon
draw.polygon([(cx - 190, 865), (cx - 135, 770), (cx + 135, 770), (cx + 190, 865), (cx + 190, 915), (cx - 190, 915)], fill=(12, 14, 26, 255), outline=(0, 245, 212, 255), width=6)
draw.polygon([(cx - 105, 775), (cx - 70, 725), (cx + 70, 725), (cx + 105, 775)], fill=(24, 34, 66, 255))
draw.rounded_rectangle([cx - 165, 832, cx - 85, 858], radius=8, fill=(255, 42, 85, 255))
draw.rounded_rectangle([cx + 85, 832, cx + 165, 858], radius=8, fill=(255, 42, 85, 255))

out_path = "/content/icon.png"
img.save(out_path)
print("Đã tạo icon:", out_path)

# Nếu dự án đã clone sẵn trên Colab, gắn luôn icon vào Android
proj = "/content/Midnight-Expressway"
if os.path.isdir(os.path.join(proj, "android")):
    os.makedirs(os.path.join(proj, "resources"), exist_ok=True)
    shutil.copy2(out_path, os.path.join(proj, "resources", "icon.png"))
    cmd = (
        f"cd {proj} && "
        "npm install --no-audit --no-fund @capacitor/assets && "
        "npx --yes @capacitor/assets generate --android --assetPath resources --iconBackgroundColor '#090b1a' --splashBackgroundColor '#090b1a' && "
        "npx --yes cap sync android"
    )
    subprocess.run(["bash", "-c", cmd], check=True)
    print("Đã cập nhật icon vào Android project!")

display(img.resize((256, 256)))


In [ ]:
import subprocess
from google.colab import files

script = r"""
set -e
export DEBIAN_FRONTEND=noninteractive

echo "=== 1. Cài OpenJDK 17 & Node.js 20 ==="
if [ ! -d "/usr/lib/jvm/java-17-openjdk-amd64" ]; then
  apt-get update -qq
  apt-get install -y -qq openjdk-17-jdk unzip wget curl
fi
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export PATH=$JAVA_HOME/bin:$PATH
java -version
if ! node -v 2>/dev/null | grep -qE "^v(20|22)\."; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  apt-get install -y -qq nodejs
fi

echo "=== 2. Cài Android SDK (API 34) ==="
export ANDROID_HOME=/opt/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools
if [ ! -d "$ANDROID_HOME/platforms/android-34" ]; then
  mkdir -p $ANDROID_HOME/cmdline-tools
  if [ ! -d "$ANDROID_HOME/cmdline-tools/latest" ]; then
    wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdline.zip
    unzip -q -o /tmp/cmdline.zip -d $ANDROID_HOME/cmdline-tools
    mv $ANDROID_HOME/cmdline-tools/cmdline-tools $ANDROID_HOME/cmdline-tools/latest
  fi
  yes | sdkmanager --sdk_root=$ANDROID_HOME --licenses >/dev/null 2>&1 || true
  yes | sdkmanager --sdk_root=$ANDROID_HOME "platform-tools" "platforms;android-34" "build-tools;34.0.0"
fi

echo "=== 3. Clone GitHub & cài Capacitor 6 ==="
cd /content
rm -rf Midnight-Expressway
git clone https://github.com/spdatdd/vibe-code-Midnight-Expressway.git Midnight-Expressway
cd Midnight-Expressway

npm install --no-audit --no-fund @capacitor/core@6 @capacitor/cli@6 @capacitor/android@6
npx --yes cap init "Midnight Expressway" "com.midnight.expressway" --web-dir dist
node -e 'const fs=require("fs");const c=JSON.parse(fs.readFileSync("capacitor.config.json","utf8"));c.server={androidScheme:"https"};fs.writeFileSync("capacitor.config.json",JSON.stringify(c,null,2));'
npx --yes cap add android

if [ ! -f resources/icon.png ] && [ -f /content/icon.png ]; then
  mkdir -p resources
  cp /content/icon.png resources/icon.png
fi
if [ -f resources/icon.png ]; then
  echo "=== 3b. Tạo icon app từ resources/icon.png ==="
  npm install --no-audit --no-fund @capacitor/assets
  npx --yes @capacitor/assets generate --android --assetPath resources --iconBackgroundColor "#090b1a" --splashBackgroundColor "#090b1a"
fi

echo "=== 4. Cấu hình Fullscreen (ẩn thanh trạng thái + thanh điều hướng) ==="
cat << 'EOF' > android/app/src/main/java/com/midnight/expressway/MainActivity.java
package com.midnight.expressway;

import android.os.Build;
import android.os.Bundle;
import android.view.WindowManager;
import androidx.core.view.WindowCompat;
import androidx.core.view.WindowInsetsCompat;
import androidx.core.view.WindowInsetsControllerCompat;
import com.getcapacitor.BridgeActivity;

public class MainActivity extends BridgeActivity {
    @Override
    protected void onCreate(Bundle savedInstanceState) {
        super.onCreate(savedInstanceState);
        if (Build.VERSION.SDK_INT >= Build.VERSION_CODES.P) {
            WindowManager.LayoutParams lp = getWindow().getAttributes();
            lp.layoutInDisplayCutoutMode =
                WindowManager.LayoutParams.LAYOUT_IN_DISPLAY_CUTOUT_MODE_SHORT_EDGES;
            getWindow().setAttributes(lp);
        }
        hideSystemBars();
    }

    @Override
    public void onWindowFocusChanged(boolean hasFocus) {
        super.onWindowFocusChanged(hasFocus);
        if (hasFocus) {
            hideSystemBars();
        }
    }

    private void hideSystemBars() {
        WindowCompat.setDecorFitsSystemWindows(getWindow(), false);
        WindowInsetsControllerCompat controller =
            WindowCompat.getInsetsController(getWindow(), getWindow().getDecorView());
        if (controller != null) {
            controller.hide(WindowInsetsCompat.Type.systemBars());
            controller.setSystemBarsBehavior(
                WindowInsetsControllerCompat.BEHAVIOR_SHOW_TRANSIENT_BARS_BY_SWIPE
            );
        }
    }
}
EOF

sed -i 's|<item name="windowNoTitle">true</item>|<item name="windowNoTitle">true</item>\n        <item name="android:windowFullscreen">true</item>|' android/app/src/main/res/values/styles.xml
npx --yes cap sync android

echo "=== 5. Build APK ==="
cd android
chmod +x gradlew
./gradlew assembleDebug --no-daemon -Dorg.gradle.java.home=$JAVA_HOME
echo "BUILD DONE: /content/Midnight-Expressway/android/app/build/outputs/apk/debug/app-debug.apk"
"""

proc = subprocess.Popen(["bash", "-c", script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Build failed with exit code {ret}")

files.download("/content/Midnight-Expressway/android/app/build/outputs/apk/debug/app-debug.apk")
